In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from scipy.stats import norm
from scipy.optimize import minimize
from copy import deepcopy
from sklearn.linear_model import LinearRegression
import torch
import torch.nn as nn
#import scipy linear regression
import statsmodels.api as sm
from scipy.stats import t as t_dist
import numpy as np
from scipy.stats import ttest_ind

### Calculate OR value

#### change with anchor mean

In [ ]:
# parameters setting
K = 5
T = 1000
N = 10
Z = t_dist.ppf(0.975,N)
std_niose = 3
anchor_set = [1,2,3,4,5]
N = K + N

cost = np.zeros((len(anchor_set),3,T))
for idx,anchor_mean in enumerate(anchor_set):
    for t in tqdm(range(T)):
        true_tao = np.random.normal(anchor_mean, 3, K )
        constant = np.random.normal(anchor_mean, 3)
        optimal_cost = np.sum(true_tao[np.argwhere(true_tao>0)])

        # generate data
        feature = np.ones((N, K+1)) 
        feature[:, 1:] = np.random.binomial(1, 0.5, (N,K))

        # generate label
        label = constant + np.dot(feature[:,1:], true_tao) + np.random.normal(0, std_niose, N)

        #-------------DM Method----------------
        X = np.zeros((K,N))
        Y = np.zeros((K,N))
        for k in range(K):
            X[k,:] = feature[:,k+1]
            Y[k,:] = label
        

        #DM
        tao_hat = np.zeros(K)
        variance = np.zeros(K)
        #variance1 = np.zeros(K)
        p_value_list = np.zeros(K)
        for k in range(K):
            group_1 = Y[k,X[k,:]==1]
            group_0 = Y[k,X[k,:]==0]
            t_stat, p_value = ttest_ind(group_1, group_0, equal_var = False)  
            diff_mean = group_1.mean() - group_0.mean()
            tao_hat[k] = diff_mean
            p_value_list[k] = p_value
            variance[k] = N*(group_1.var(ddof=1) / len(group_1) + group_0.var(ddof=1) / len(group_0))
        #estimated cost
        decision1 = np.argwhere(tao_hat>(Z*np.sqrt(variance)))
        if optimal_cost == 0:
            if len(decision1) == 0:
                cost[idx,0,t] = 1
            else:
                cost[idx,0,t] = 0
        else:
            cost[idx,0,t] = np.sum(true_tao[decision1])/optimal_cost
        

        tao_0 = np.mean(tao_hat)
        numerator = np.mean(variance)
        denumerator = np.mean((tao_hat - tao_0)**2) - numerator/N

        bayesian_tao = np.zeros(K)
        bayesian_beta = np.zeros(K)

        decision3 = []
  
        for k in range(K):
            if denumerator <= 0:
                theta = 1
                posteri_mean = tao_hat[k]*theta + (1 - theta)*tao_0
                posteri_var = 1/(N/variance[k])
                dist = norm(loc=posteri_mean, scale=np.sqrt(posteri_var))
                prob = dist.sf(0)  # survival function: P(X > x)
                if prob > 1 - 0.025:
                    decision3.append(k)
            else:
                bayesian_beta[k] = max(variance[k]/denumerator,0)
                theta = N/(N+bayesian_beta[k])
                posteri_mean = tao_hat[k]*theta + (1 - theta)*tao_0
                posteri_var = 1/(1/denumerator+ N/variance[k])
                dist = norm(loc=posteri_mean, scale=np.sqrt(posteri_var))

                prob = dist.sf(0)  # survival function: P(X > x)
                if prob > 1 - 0.025:
                    decision3.append(k)

        if optimal_cost == 0:
            if len(decision3) == 0:
                cost[idx,2,t] = 1
            else:
                cost[idx,2,t] = 0
        else:
            cost[idx,2,t] = np.sum(true_tao[decision3])/optimal_cost

        
        
        if tao_0 == 0:
            cost[idx,1,t] = cost[idx,0,t]
            continue
        beta = numerator/denumerator + Z*np.sqrt(N*numerator)/tao_0
        beta = max(0,beta)
        theta = N/(N+beta)
        #shunken_saa
        tao_shunken_hat = np.zeros(K)
        p_value_list_shrunken = np.zeros(K)
        Y_shunken = deepcopy(Y)
        Y_shunken = theta*Y_shunken 
        for k in range(K):
            group_1 = Y_shunken[k,X[k,:]==1] + (1-theta)*tao_0
            group_0 = Y_shunken[k,X[k,:]==0]
            t_stat, p_value = ttest_ind(group_1, group_0, equal_var = False)  
            diff_mean1 = group_1.mean() - group_0.mean()
            tao_shunken_hat[k] = diff_mean1
            p_value_list_shrunken[k] = p_value


        decision2 = np.intersect1d(np.argwhere(p_value_list_shrunken<0.05), np.argwhere(tao_shunken_hat>0))

        if optimal_cost == 0:
            if len(decision2) == 0:
                cost[idx,1,t] = 1
            else:
                cost[idx,1,t] = 0
        else:
            cost[idx,1,t] = np.sum(true_tao[decision2])/optimal_cost